# Notebook 03: Professional Feature-Selection Funnel (A to H)

This notebook implements each funnel stage with theory + experiments:
A) Variance threshold
B) Correlation filtering (Pearson/Spearman)
C) Model-based importance
D) Permutation importance
E) RFE / RFECV
F) L1 regularization
G) Mutual information
H) SHAP-based feature ranking


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from src.data_loader import load_isolet_dataset
from src.feature_selector import FeatureSelector
from src.visualization import (
    plot_correlation_heatmap,
    plot_feature_importance,
    plot_feature_selection_funnel,
    plot_mutual_information,
    plot_permutation_importance,
    plot_shap_bar,
    plot_variance_distribution,
)

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid')
Path('outputs/figures').mkdir(parents=True, exist_ok=True)


In [ ]:
X, y, _ = load_isolet_dataset()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_train_fs, X_val_fs, y_train_fs, y_val_fs = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Use a bounded subset so expensive selector stages finish in notebook runtime.
subset_size = min(2200, len(X_train_fs))
X_train_small, _, y_train_small, _ = train_test_split(
    X_train_fs,
    y_train_fs,
    train_size=subset_size,
    random_state=42,
    stratify=y_train_fs,
)

print('ISOLET full shape:', X.shape)
print('Train small shape:', X_train_small.shape)
print('Validation shape:', X_val_fs.shape)
print('Test shape:', X_test.shape)


## Baseline before funnel

We measure quality before filtering so every stage has a reference point.


In [ ]:
baseline_model = RandomForestClassifier(n_estimators=160, random_state=42, n_jobs=-1)
baseline_model.fit(X_train_small, y_train_small)
baseline_acc = accuracy_score(y_test, baseline_model.predict(X_test))
print(f'Baseline accuracy with {X_train_small.shape[1]} features: {baseline_acc:.4f}')


## Stage A: Variance Threshold

**Definition:** Remove features with too little spread.

**Mathematical intuition:** If variance is near zero, the feature carries little discriminative information.


In [ ]:
variance_thresholds = [0.0, 0.001, 0.01, 0.05]
variance_rows = []
for thr in variance_thresholds:
    vt = VarianceThreshold(threshold=thr)
    Xt = vt.fit_transform(X_train_small)
    variance_rows.append({'threshold': thr, 'features_kept': Xt.shape[1]})

variance_df = pd.DataFrame(variance_rows)
variance_df


In [ ]:
selector = FeatureSelector(random_state=42)
selector._check_data(X_train_small, y_train_small)
selector.variance_threshold(threshold=0.01)
var_info = selector.results_['variance_threshold']
variance_series = pd.Series(var_info['variances'])
plot_variance_distribution(variance_series, threshold=0.01)


## Stage B: Correlation Filtering (Pearson vs Spearman)

**Definition:** Remove highly correlated pairs to reduce redundancy and multicollinearity.


In [ ]:
corr_rows = []
for method in ['pearson', 'spearman']:
    for threshold in [0.80, 0.90, 0.95]:
        selector_tmp = FeatureSelector(random_state=42)
        selector_tmp.variance_threshold(X_train_small, threshold=0.01)
        selector_tmp.correlation_filter(method=method, threshold=threshold)
        corr_rows.append(
            {
                'method': method,
                'threshold': threshold,
                'features_kept': len(selector_tmp.selected_features_),
            }
        )

corr_df = pd.DataFrame(corr_rows)
corr_df


In [ ]:
selector.correlation_filter(method='spearman', threshold=0.90)
corr_matrix = X_train_small[selector.selected_features_].corr(method='spearman').abs()
plot_correlation_heatmap(corr_matrix.iloc[:80, :80], title='Stage B Spearman Correlation Heatmap')
print(f'After Stage B: {len(selector.selected_features_)} features')


## Stages C and D: Model Importance and Permutation Importance

**Tradeoff:** Tree impurity importance is fast but can be biased. Permutation importance is slower but more faithful to predictive dependency.


In [ ]:
selector.model_importance()
rf_importance = selector.results_['model_importance']['feature_importances']
plot_feature_importance(rf_importance, top_n=25, title='Stage C - Random Forest Importance')

selector.permutation_importance(X_eval=X_val_fs, y_eval=y_val_fs, n_repeats=5, n_features_to_show=25)
perm_df = selector.results_['permutation_importance']['top_features']
plot_permutation_importance(perm_df, top_n=25)


## Stage E: RFE / RFECV

**Definition:** Recursively remove low-importance features and optionally use CV to pick feature count.


In [ ]:
# Fixed-count RFE for the primary funnel.
selector.rfe(n_features_to_select=180, step=20, use_cv=False)
print(f'After Stage E (RFE): {len(selector.selected_features_)} features')

# RFECV demonstration on a reduced candidate pool to keep runtime bounded.
rfecv_candidates = selector.selected_features_[:120]
rfecv_selector = FeatureSelector(random_state=42)
rfecv_selector.rfe(
    X=X_train_small[rfecv_candidates],
    y=y_train_small,
    n_features_to_select=60,
    step=15,
    model=RandomForestClassifier(n_estimators=60, random_state=42, n_jobs=-1),
    use_cv=True,
    min_features_to_select=30,
)
print(
    f"RFECV demo selected {len(rfecv_selector.selected_features_)} features "
    f"from {len(rfecv_candidates)} candidates"
)


## Stage F + G + H: L1, Mutual Information, and SHAP


In [ ]:
selector.l1_selection(C=0.5, max_iter=2500)
print(f'After Stage F (L1): {len(selector.selected_features_)} features')

selector.mutual_information(k=120)
mi_df = selector.results_['mutual_information']['top_features']
plot_mutual_information(mi_df, top_n=25)
print(f'After Stage G (MI): {len(selector.selected_features_)} features')

selector.shap_selection(k=90, background_samples=30)
shap_df = selector.results_['shap_selection']['top_features']
plot_shap_bar(shap_df, top_n=25)
print(f'After Stage H (SHAP): {len(selector.selected_features_)} features')


In [ ]:
selected_features = selector.selected_features_
X_train_selected = X_train_small[selected_features]
X_test_selected = X_test[selected_features]

final_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
final_model.fit(X_train_selected, y_train_small)
final_acc = accuracy_score(y_test, final_model.predict(X_test_selected))

stage_counts = {
    'Initial': X_train_small.shape[1],
    'A: Variance': selector.results_['variance_threshold']['features_kept'],
    'B: Correlation': selector.results_['correlation_filter']['features_kept'],
    'E: RFECV': selector.results_['rfe']['features_kept'],
    'F: L1': selector.results_['l1_selection']['features_kept'],
    'G: MI': selector.results_['mutual_information']['features_kept'],
    'H: SHAP': selector.results_['shap_selection']['features_kept'],
}
plot_feature_selection_funnel(stage_counts)

pd.DataFrame(
    {
        'metric': ['baseline_accuracy', 'post_funnel_accuracy', 'feature_reduction_pct'],
        'value': [
            baseline_acc,
            final_acc,
            (1 - len(selected_features) / X_train_small.shape[1]) * 100,
        ],
    }
)


## Interpretation

- The funnel progressively reduces dimensions while preserving strong predictive signal.
- Different selectors contribute differently: filtering stages remove obvious noise/redundancy, while embedded/SHAP stages refine ranking.
- This staged approach is more stable than applying one aggressive method directly.
